## Prepare requierments

In [ ]:
pip uninstall -y transformers


Found existing installation: transformers 5.2.0
Uninstalling transformers-5.2.0:
  Successfully uninstalled transformers-5.2.0


In [1]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 13.7 MB/s eta 0:00:00


In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers
!pip install peft
!pip install accelerate datasets bitsandbytes

Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached transformers-5.2.0-py3-none-any.whl.metadata (32 kB)
Using cached transformers-5.2.0-py3-none-any.whl (10.4 MB)


## import Libraries

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, pipeline
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
import torch
import os
from peft import LoraConfig, get_peft_model

In [3]:
print("CUDA:", torch.cuda.is_available())
print("TRL OK")

print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory/1e9)

CUDA: True
TRL OK
Tesla T4
15.637086208


## load Dataset

In [4]:
from datasets import load_dataset

data = load_dataset("sahil2801/CodeAlpaca-20k", split="train")
len(data)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/147 [00:00<?, ?B/s]

code_alpaca_20k.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/20022 [00:00<?, ? examples/s]

20022

In [30]:
dataset = data

In [31]:
for i in range(2): # 3
  for k, v in dataset[i].items():
    print(k, ":", v)
  print("-----"*10)

output : arr = [2, 4, 6, 8, 10]
instruction : Create an array of length 5 which contains all even numbers between 1 and 10.
input : 
--------------------------------------------------
output : Height of triangle = opposite side length * sin (angle) / side length
instruction : Formulate an equation to calculate the height of a triangle given the angle, side lengths and opposite side length.
input : 
--------------------------------------------------


In [32]:
keywords = [
    # Libraries
    "matplotlib", "seaborn", "pandas", "numpy",

    # Visualization
    "plot", "visualize", "graph", "chart",
    "histogram", "bar", "line", "scatter",
    "distribution", "boxplot", "heatmap",

    # Data terms
    "dataframe", "dataset", "csv",
    "analyze", "analysis", "statistics",
    "mean", "median", "correlation"
]

In [33]:
def filter_visualization(example):
    text = (
        example["instruction"] +
        example["input"] +
        example["output"]
    ).lower()

    return any(keyword in text for keyword in keywords)

In [35]:
dataset = dataset.filter(filter_visualization)

Filter:   0%|          | 0/1508 [00:00<?, ? examples/s]

In [37]:
import pandas as pd

df = dataset.to_pandas()
print(df.isna().sum())

n = len(dataset)
print(f"Number of samples after filter : {n}")

output         0
instruction    0
input          0
dtype: int64
Number of samples after filter : 1508


In [11]:
def preprocess_function(example):

    if example["input"].strip() != "":
        user_content = (
            f"{example['instruction']}\n\n"
            f"{example['input']}"
        )
    else:
        user_content = example["instruction"]

    return {
        "prompt": [
            {
                "role": "user",
                "content": user_content,
            }
        ],
        "completion": [
            {
                "role": "assistant",
                "content": example["output"],
            }
        ],
    }


In [38]:
def preprocess_function(example):
    if example["input"].strip() != "":
        user_content = (
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Input:\n{example['input']}"
        )
    else:
        user_content = f"### Instruction:\n{example['instruction']}"

    return {
        "text": (
            f"{user_content}\n\n"
            f"### Response:\n{example['output']}"
        )
    }

In [39]:
dataset = dataset.map(
    preprocess_function,
    remove_columns=dataset.column_names)

Map:   0%|          | 0/1508 [00:00<?, ? examples/s]

In [40]:
dataset = dataset.select(range(800))

In [41]:
for i in range(2): # 3

    for k, v in dataset[i].items():
        print(k, ":", v)
        print("-----"*3)
    print("===="*20)

text : ### Instruction:
Develop a classification algorithm in Python to predict whether a bird is a hawk or a falcon.

### Response:
import pandas as pd
import numpy as np

# Define features
features = ["wing_shape", "size", "tails_length", "color_pattern"] 

# Load the data
data = pd.read_csv("birds.csv")

# Preprocess the data to generate feature vectors
X = np.array(data[features]) 

# Make a target vector 
y = np.array(data["species"]) 

# Divide the data into training and test sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a classification model
from sklearn.tree import DecisionTreeClassifier
model = DecisionTreeClassifier()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

# Generate evaluation metrics
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)

print("Model accuracy: %.2f" % accuracy)
---------------

## Get model

In [42]:
model_name = "HuggingFaceTB/SmolLM3-3B"

model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path=model_name,
                                             device_map="auto",
                                             torch_dtype=torch.float16)

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

## Training config

In [43]:
# Load model with PEFT config
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj"
    ]
)

In [48]:
config = SFTConfig(
    output_dir="./smollm3-visualization-sft",
    per_device_train_batch_size=4,
    learning_rate=5e-4,
    num_train_epochs=2,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=True,
    dataset_text_field="text"
)


In [17]:
import trl
print(trl.__version__)

0.29.0


In [49]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=config,
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


## Training

In [46]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [50]:
trainer.train()

Step,Training Loss
10,1.243429
20,0.933516
30,0.922671
40,0.933304
50,0.980660
60,0.839292
70,0.943999
80,0.876877
90,0.913695
100,0.854357


TrainOutput(global_step=400, training_loss=0.8332025277614593, metrics={'train_runtime': 381.98, 'train_samples_per_second': 4.189, 'train_steps_per_second': 1.047, 'total_flos': 4708072765390848.0, 'train_loss': 0.8332025277614593})

## Save New model - Push to Hugging face hub

In [63]:
trainer.save_model("./smollm3-visualization-sft/final")
tokenizer.save_pretrained("./smollm3-visualization-sft/final")
print("Model saved!")

Model saved!


In [71]:
trainer.model.save_pretrained("lora_adapter")
tokenizer.save_pretrained("lora_adapter")

('lora_adapter/tokenizer_config.json',
 'lora_adapter/chat_template.jinja',
 'lora_adapter/tokenizer.json')

In [64]:
from google.colab import userdata
HF_token = userdata.get('Hugging_face_tokens')

In [65]:
from huggingface_hub import HfApi
import os
api = HfApi(token= HF_token)


In [66]:
from huggingface_hub import login
login(token= HF_token)

In [67]:
from huggingface_hub import whoami
whoami()

{'type': 'user',
 'id': '68a4ea54e9eadb6d66ea196b',
 'name': 'Kyrollos32',
 'fullname': 'Kyrollos Ashraf',
 'email': 'kerlosashraf33@gmail.com',
 'emailVerified': True,
 'canPay': False,
 'billingMode': 'prepaid',
 'periodEnd': 1775001600,
 'isPro': False,
 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/no-auth/VUQgWdfh2qX3IAygMyuAN.png',
 'orgs': [],
 'auth': {'type': 'access_token',
  'accessToken': {'displayName': 'kero2',
   'role': 'write',
   'createdAt': '2026-03-01T19:14:22.371Z'}}}

In [68]:
from huggingface_hub import HfApi
api = HfApi()

api.create_repo(
    repo_id="Kyrollos32/smollm_sft_2",
    repo_type="model",
    exist_ok=True
)

RepoUrl('https://huggingface.co/Kyrollos32/smollm_sft_2', endpoint='https://huggingface.co', repo_type='model', repo_id='Kyrollos32/smollm_sft_2')

In [73]:
api.upload_folder(
    folder_path="lora_adapter",
    repo_id="Kyrollos32/smollm_sft_2",
    repo_type="model",
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ra_adapter/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

  ...adapter_model.safetensors:   4%|3         |  556kB / 15.4MB            

CommitInfo(commit_url='https://huggingface.co/Kyrollos32/smollm_sft_2/commit/13bdaed36447a73738f94a98ed348af7d956ba3f', commit_message='Upload folder using huggingface_hub', commit_description='', oid='13bdaed36447a73738f94a98ed348af7d956ba3f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Kyrollos32/smollm_sft_2', endpoint='https://huggingface.co', repo_type='model', repo_id='Kyrollos32/smollm_sft_2'), pr_revision=None, pr_num=None)

In [ ]:
trainer.push_to_hub("Kyrollos32/smollm3-CodeGenerator")
tokenizer.push_to_hub("Kyrollos32/smollm3-CodeGenerator")


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...-sft/final/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

  ...zation-sft/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

  ...adapter_model.safetensors:   2%|1         |  280kB / 15.4MB            

...

  ...ion-sft/training_args.bin:   2%|1         |   101B / 5.58kB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpdr49oku_/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

CommitInfo(commit_url='https://huggingface.co/Kyrollos32/smollm3-CodeGenerator/commit/703827fea7ab214a7d339f49df4e0a4e738aca32', commit_message='Upload tokenizer', commit_description='', oid='703827fea7ab214a7d339f49df4e0a4e738aca32', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Kyrollos32/smollm3-CodeGenerator', endpoint='https://huggingface.co', repo_type='model', repo_id='Kyrollos32/smollm3-CodeGenerator'), pr_revision=None, pr_num=None)

In [74]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM3-3B",
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM3-3B")

model = PeftModel.from_pretrained(
    base_model,
    "Kyrollos32/smollm_sft_2"
)

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/15.4M [00:00<?, ?B/s]

In [75]:
model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): SmolLM3ForCausalLM(
      (model): SmolLM3Model(
        (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
        (layers): ModuleList(
          (0-35): 36 x SmolLM3DecoderLayer(
            (self_attn): SmolLM3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
         

In [76]:
prompt = """
Write Python code using matplotlib to create a line plot
for a DataFrame column named 'sales'.
"""

In [77]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.2,
        top_p=0.9
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)


Write Python code using matplotlib to create a line plot
for a DataFrame column named'sales'.
# Import the pandas library
import pandas as pd

# Import the matplotlib library
import matplotlib.pyplot as plt

# Load the data into a DataFrame
df = pd.read_csv('data.csv')

# Create a line plot for the'sales' column
plt.plot(df['sales'])

# Add a title and labels
plt.title('Sales')
plt.xlabel('Date')
plt.ylabel('Sales')

# Show the plot
plt.show()


In [78]:
response = response[len(prompt):]
print(response)

 Import the pandas library
import pandas as pd

# Import the matplotlib library
import matplotlib.pyplot as plt

# Load the data into a DataFrame
df = pd.read_csv('data.csv')

# Create a line plot for the'sales' column
plt.plot(df['sales'])

# Add a title and labels
plt.title('Sales')
plt.xlabel('Date')
plt.ylabel('Sales')

# Show the plot
plt.show()


In [79]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

result = pipe(prompt, max_new_tokens=200)
print(result[0]["generated_text"])

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Write Python code using matplotlib to create a line plot
for a DataFrame column named 'sales'.
### Input ###
import pandas as pd
import matplotlib.pyplot as plt

# Read the data
df = pd.read_csv("sales_data.csv")

### Response ###
plt.figure(figsize=(10, 6))
plt.plot(df['sales'])
plt.xlabel('Date')
plt.ylabel('Sales')
plt.title('Sales over Time')
plt.show()


# Merge and push

In [80]:
merged_model = model.merge_and_unload()

In [81]:
merged_model.save_pretrained("smollm_sft_2_full")
tokenizer.save_pretrained("smollm_sft_2_full")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('smollm_sft_2_full/tokenizer_config.json',
 'smollm_sft_2_full/chat_template.jinja',
 'smollm_sft_2_full/tokenizer.json')

In [82]:
api.create_repo(
    repo_id="Kyrollos32/smollm_sft_2_full",
    repo_type="model",
    exist_ok=True
)

RepoUrl('https://huggingface.co/Kyrollos32/smollm_sft_2_full', endpoint='https://huggingface.co', repo_type='model', repo_id='Kyrollos32/smollm_sft_2_full')

In [83]:
api.upload_folder(
    folder_path="smollm_sft_2_full",
    repo_id="Kyrollos32/smollm_sft_2_full",
    repo_type="model",
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...sft_2_full/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

  ..._2_full/model.safetensors:   0%|          | 23.9MB / 6.15GB            

CommitInfo(commit_url='https://huggingface.co/Kyrollos32/smollm_sft_2_full/commit/baa2155489c63cca0177ac53732d81154044e13f', commit_message='Upload folder using huggingface_hub', commit_description='', oid='baa2155489c63cca0177ac53732d81154044e13f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Kyrollos32/smollm_sft_2_full', endpoint='https://huggingface.co', repo_type='model', repo_id='Kyrollos32/smollm_sft_2_full'), pr_revision=None, pr_num=None)

In [84]:
# How to use it

In [84]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model = AutoModelForCausalLM.from_pretrained(
    "Kyrollos32/smollm_sft_2_full",
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(
    "Kyrollos32/smollm_sft_2_full"
)

model.eval()